# Zadanie 5: programowanie genetyczne i regresja symboliczna
markdown
#VSC-20d46ce3
markdown
# Zadanie na 4.0

Porównuję te same trzy konfiguracje na próbkach losowanych z szerszego zakresu $[-15, 15]$ oraz dla szumu o odchyleniu standardowym 2 i 5.
code
#VSC-54c31648
python
np.random.seed(0)
X_wide = np.random.uniform(-15, 15, size=(200, 6))
y_wide = 2.2 * np.sin(X_wide[:, 0] + 2 * X_wide[:, 1]) - X_wide[:, 5]**2 - 3

configs = [
    dict(
        binary_operators=["+", "*"],
        unary_operators=["cos", "exp", "sin"],
        maxsize=20,
    ),
    dict(
        binary_operators=["+", "*", "-", "my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
        unary_operators=["cos", "exp", "sin", "log"],
        maxsize=30,
        constraints={"my_pow": (-1, 1)},
        extra_sympy_mappings={"my_pow": lambda x, y: x**y},
    ),
    dict(
        binary_operators=["+", "*", "-", "my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
        unary_operators=["exp", "sin"],
        maxsize=15,
        constraints={"my_pow": (-1, 1)},
        extra_sympy_mappings={"my_pow": lambda x, y: x**y},
    ),
]

for noise_std in [0.0, 2.0, 5.0]:
    print(f"\nSzum std = {noise_std}")
    y_train = y_wide + noise_std * np.random.randn(len(y_wide))
    for config_index, config in enumerate(configs, start=1):
        model = PySRRegressor(
            niterations=100,
            populations=30,
            model_selection="best",
            verbosity=False,
            **config,
        )
        model.fit(X_wide, y_train)
        top_3_scores = model.equations_.sort_values("score", ascending=False).head(3)
        print(f"Konfiguracja {config_index}:")
        for rank, (_, row) in enumerate(top_3_scores.iterrows(), start=1):
            print(f"  {rank}. score={row['score']:.4f}, wzór={row['sympy_format']}")
        print(f"  best: {model.sympy()}")
        print("-" * 30)
2. Zanotuj wzory trzech rozwiązań o najwyższej wartości `score` oraz rozwiązanie `best` dla następujących zestawów ustawień:
   1. `binary_operators=["+", "*"], unary_operators=["cos", "exp", "sin"], maxsize=20`,
   2. `binary_operators=["+", "*", "-", "^"], unary_operators=["cos", "exp", "sin", "log"], maxsize=30`, (dodaj ograniczenie dla argumentów operatora "^": [https://astroautomata.com/PySR/v1.5.9/options.html#constraining-use-of-operators](https://astroautomata.com/PySR/v1.5.9/options.html#constraining-use-of-operators).
   3. `binary_operators=["+", "*", "-", "^"], unary_operators=["exp", "sin"], maxsize=15`.
3. Powtórz eksperymenty z zadania na 3.0 po dodaniu szumu do próbek z funkcji $f$ (rozkład normalny o średniej 0 i odchyleniu standardowym 0.5)

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Dodaj do porównania dopasowanie oparte o próbki losowane w szerszym zakresie (między -15 a 15) oraz wyższy poziom szumu (odchylenie standardowe równe 2 oraz 5).

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zamień funkcję $f$ na $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$ gdzie $p(i)$ oznacza $i$-tą liczbę pierwszą. Uwzględnij `p` jako dodatkowy operator unarny analogicznie do przykładu "Julia packages and types" z notatnika `pysr_demo.ipynb`. Powtórz eksperymenty opisane w zadaniach na 3.0 i 4.0.


# Zadanie 1 (Na 3.0)

## Ekspeymenty bez sumu

In [ ]:
import pysr
import sympy
import numpy as np
from matplotlib import pyplot as plt
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split

np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))
y = 2.2 * np.sin(X[:, 0]+ 2*X[:,1]) - X[:,5]**2 - 3

ModuleNotFoundError: No module named 'sympy'

In [ ]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    verbosity=False,
    **default_pysr_params,
)
model.fit(X, y)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 11 (Score: 13.7280):
Wzór: x5*x5*(-1.0) + sin(x0 + x1 + x1)*2.2 - 3.0
------------------------------
Miejsce 4 (Score: 2.0312):
Wzór: x5*(-1.1931677)*x5
------------------------------
Miejsce 10 (Score: 1.1394):
Wzór: x5*(-0.9899335)*x5 + sin(x0 + x1 + x1) - 3.0889792
------------------------------


x5*x5*(-1.0) + sin(x0 + x1 + x1)*2.2 - 3.0

In [ ]:
model = PySRRegressor(
    niterations=100,
    binary_operators=[
        "+", "*", "-",
        "my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"
    ],
    extra_sympy_mappings={
        "my_pow": lambda x, y: x**y
    },
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    verbosity=False,
    constraints={'my_pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 11 (Score: 27.2189):
Wzór: -x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0
------------------------------
Miejsce 4 (Score: 3.1427):
Wzór: -x5*x5 - 3.0129473
------------------------------
Miejsce 9 (Score: 1.1454):
Wzór: -x5*x5 + sin(x0 + x1 + x1) - 3.0070155
------------------------------


-x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0

In [ ]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-","my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
    extra_sympy_mappings={
        "my_pow": lambda x, y: x**y
    },
    unary_operators=["exp", "sin"],
    maxsize=15,
    verbosity=False,
    constraints={'my_pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 11 (Score: 27.2399):
Wzór: -x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0000002
------------------------------
Miejsce 4 (Score: 3.2011):
Wzór: -x5*x5 - 3.0129473
------------------------------
Miejsce 9 (Score: 1.1391):
Wzór: -x5*x5 + sin(x0 + x1*2.002528) - 3.006651
------------------------------


-x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0000002

## Eksperymenty z szumem

In [ ]:
np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))
y = 2.2 * np.sin(X[:, 0]+ 2*X[:,1]) - X[:,5]**2 - 3
noise = 0.5 * np.random.randn(200)
y_noised = y + noise

In [ ]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    verbosity=False,
    **default_pysr_params,
)
model.fit(X, y_noised)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 4 (Score: 2.0183):
Wzór: x5*x5*(-1.1879632)
------------------------------
Miejsce 12 (Score: 1.3250):
Wzór: x5*(-0.99840987)*x5 + sin(x0 + x1 + x1)*2.215479 - 2.9473767
------------------------------
Miejsce 10 (Score: 0.9533):
Wzór: x5*(-0.98821336)*x5 + sin(x0 + x1 + x1) - 3.0375047
------------------------------


x5*(-0.99840987)*x5 + sin(x0 + x1 + x1)*2.215479 - 2.9473767

In [ ]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-","my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
    extra_sympy_mappings={
        "my_pow": lambda x, y: x**y
    },
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    verbosity=False,
    constraints={'pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y_noised)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 4 (Score: 3.0386):
Wzór: -x5*x5 - 2.9474645
------------------------------
Miejsce 9 (Score: 0.9575):
Wzór: -(x5*x5 + sin(-x0 + x1*(-1.9819049))) - 2.9439185
------------------------------
Miejsce 10 (Score: 0.7377):
Wzór: -(x5*x5 + sin(-x0 + x1*(-1.9806921))*2.2229629) - 2.9401243
------------------------------


-(x5*x5 + sin(-x0 + x1*(-1.9806921))*2.2229629) - 2.9401243

In [ ]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-","my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
    extra_sympy_mappings={
        "my_pow": lambda x, y: x**y
    },
    unary_operators=["exp", "sin"],
    maxsize=15,
    verbosity=False,
    constraints={'pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y_noised)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 4 (Score: 3.0967):
Wzór: -x5*x5 - 2.9474645
------------------------------
Miejsce 9 (Score: 0.9574):
Wzór: -x5*x5 + sin(x0 + x1*1.9818705) - 1*2.9438612
------------------------------
Miejsce 8 (Score: 0.0180):
Wzór: -x5*x5 + sin(sin(x1*(-2.0216112))) - 1*3.0098364
------------------------------


-x5*x5 + sin(x0 + x1*1.9818705) - 1*2.9438612

# Zadanie na 4.0

In [ ]:
np.random.seed(0)
X_wide = np.random.uniform(-15, 15, size=(200, 6))
y_wide = 2.2 * np.sin(X_wide[:, 0] + 2 * X_wide[:, 1]) - X_wide[:, 5]**2 - 3

configs = [
    dict(
        binary_operators=["+", "*"],
        unary_operators=["cos", "exp", "sin"],
        maxsize=20,
    ),
    dict(
        binary_operators=["+", "*", "-", "my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
        unary_operators=["cos", "exp", "sin", "log"],
        maxsize=30,
        constraints={"my_pow": (-1, 1)},
        extra_sympy_mappings={"my_pow": lambda x, y: x**y},
    ),
    dict(
        binary_operators=["+", "*", "-", "my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
        unary_operators=["exp", "sin"],
        maxsize=15,
        constraints={"my_pow": (-1, 1)},
        extra_sympy_mappings={"my_pow": lambda x, y: x**y},
    ),
]

for noise_std in [0.0, 2.0, 5.0]:
    print(f"\nSzum std = {noise_std}")
    y_train = y_wide + noise_std * np.random.randn(len(y_wide))
    for config_index, config in enumerate(configs, start=1):
        model = PySRRegressor(
            niterations=100,
            populations=30,
            model_selection="best",
            verbosity=False,
            **config,
        )
        model.fit(X_wide, y_train)
        top_3_scores = model.equations_.sort_values("score", ascending=False).head(3)
        print(f"Konfiguracja {config_index}:")
        for rank, (_, row) in enumerate(top_3_scores.iterrows(), start=1):
            print(f"  {rank}. score={row['score']:.4f}, wzór={row['sympy_format']}")
        print(f"  best: {model.sympy()}")
        print("-" * 30)

# Zadanie na 5.0

In [ ]:
from pysr import jl

jl.seval("""
import Pkg
Pkg.add("Primes")
""")

jl.seval("using Primes: prime")

jl.seval("""
function p(x::T) where T
    floor_val = floor(Int, x)
    if 1 <= floor_val < 1000
        return T(prime(floor_val))
    else
        return T(NaN)
    end
end
""")

print("p(1.9) =", jl.p(1.9), "(oczekiwane: 2.0, bo prime(floor(1.9))=prime(1)=2)")
print("p(3.0) =", jl.p(3.0), "(oczekiwane: 5.0, bo prime(3)=5)")
print("p(0.9) =", jl.p(0.9), "(oczekiwane: NaN, bo floor(0.9)=0 < 1)")
print("p(-1.0) =", jl.p(-1.0), "(oczekiwane: NaN, bo floor(-1.0)=-1 < 1)")



In [ ]:
class sympy_p(sympy.Function):
    pass


def compute_y_prime(X):
    """Oblicza f(x) = 2.2*sin(x0 + 2*x1) - x5^2 - p(floor(x0))."""
    base = 2.2 * np.sin(X[:, 0] + 2 * X[:, 1]) - X[:, 5]**2
    p_vals = np.array([float(jl.p(x0)) for x0 in X[:, 0]])
    return base - p_vals


def make_dataset_prime(lo, hi, n=200, noise_std=0.0, seed=0):
    """Generuje n próbek z zakresu [lo, hi]."""
    rng = np.random.RandomState(seed)
    collected_X, collected_y = [], []

    while sum(len(a) for a in collected_X) < n:
        X_batch = rng.uniform(lo, hi, size=(n * 3, 6))
        y_batch = compute_y_prime(X_batch)
        valid = ~np.isnan(y_batch)
        collected_X.append(X_batch[valid])
        collected_y.append(y_batch[valid])

    X_all = np.vstack(collected_X)[:n]
    y_all = np.concatenate(collected_y)[:n]

    if noise_std > 0:
        y_all = y_all + noise_std * rng.randn(n)

    return X_all, y_all


configs_prime = [
    dict(
        binary_operators=["+", "*"],
        unary_operators=["cos", "exp", "sin", "p"],
        maxsize=20,
        extra_sympy_mappings={"p": sympy_p},
    ),
    dict(
        binary_operators=["+", "*", "-", "my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
        unary_operators=["cos", "exp", "sin", "log", "p"],
        maxsize=30,
        constraints={"my_pow": (-1, 1)},
        extra_sympy_mappings={"my_pow": lambda x, y: x**y, "p": sympy_p},
    ),
    dict(
        binary_operators=["+", "*", "-", "my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
        unary_operators=["exp", "sin", "p"],
        maxsize=15,
        constraints={"my_pow": (-1, 1)},
        extra_sympy_mappings={"my_pow": lambda x, y: x**y, "p": sympy_p},
    ),
]

In [ ]:
for noise_std in [0.0, 0.5]:
    X_p, y_p = make_dataset_prime(-5, 5, n=200, noise_std=noise_std, seed=0)
    print(f"\n{'='*50}")
    print(f"Zakres [-5, 5], szum std = {noise_std}")
    print(f"{'='*50}")
    for config_index, config in enumerate(configs_prime, start=1):
        cfg = config.copy()
        model = PySRRegressor(
            niterations=100,
            populations=30,
            model_selection="best",
            verbosity=False,
            **cfg,
        )
        model.fit(X_p, y_p)
        top_3_scores = model.equations_.sort_values("score", ascending=False).head(3)
        print(f"Konfiguracja {config_index}:")
        for rank, (_, row) in enumerate(top_3_scores.iterrows(), start=1):
            print(f"  {rank}. score={row['score']:.4f}, wzór={row['sympy_format']}")
        print(f"  best: {model.sympy()}")
        print("-" * 30)

In [ ]:
for noise_std in [0.0, 2.0, 5.0]:
    X_p, y_p = make_dataset_prime(-15, 15, n=200, noise_std=noise_std, seed=0)
    print(f"\n{'='*50}")
    print(f"Zakres [-15, 15], szum std = {noise_std}")
    print(f"{'='*50}")
    for config_index, config in enumerate(configs_prime, start=1):
        cfg = config.copy()
        model = PySRRegressor(
            niterations=100,
            populations=30,
            model_selection="best",
            verbosity=False,
            **cfg,
        )
        model.fit(X_p, y_p)
        top_3_scores = model.equations_.sort_values("score", ascending=False).head(3)
        print(f"Konfiguracja {config_index}:")
        for rank, (_, row) in enumerate(top_3_scores.iterrows(), start=1):
            print(f"  {rank}. score={row['score']:.4f}, wzór={row['sympy_format']}")
        print(f"  best: {model.sympy()}")
        print("-" * 30)